# next-track-rec 02: SASRec retrieval

Runs code from github.com/N-Mehdi/next-track-rec on the private dataset yambda-50m-prepared.
Configuration A (embedding dim 128), up to 60 epochs with early stopping (patience 3).
The first run (A and B, 20 epochs) stopped at the epoch limit. Outputs go to /kaggle/working/outputs.

In [ ]:
import glob
import os
import subprocess

# Single GPU: the session has two T4, training uses one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO = "https://github.com/N-Mehdi/next-track-rec.git"
CODE = "/tmp/next-track-rec"
subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "."], check=True)

PREPARED = os.path.dirname(glob.glob("/kaggle/input/**/item_map", recursive=True)[0])
OUTPUTS = "/kaggle/working/outputs"
print(PREPARED)
subprocess.run(["nvidia-smi"], check=False)
check = "import torch; print('visible GPUs', torch.cuda.device_count(), torch.cuda.get_device_name(0))"
subprocess.run(["python", "-c", check], check=True)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m nexttrack.train.train_sasrec --set paths.prepared={PREPARED} paths.outputs={OUTPUTS} sasrec.run_name=a_d128_e60 sasrec.embedding_dim=128 sasrec.epochs=60 sasrec.patience=3

In [ ]:
import json

with open(os.path.join(OUTPUTS, "a_d128_e60", "report.json"), encoding="utf-8") as handle:
    report = json.load(handle)
print("best epoch", report["best_epoch"], "train seconds", round(report["train_seconds"]))
print("val ", report["validation"]["all"]["ndcg@10"], report["validation"]["all"]["recall@10"])
print("test", report["test"]["all"]["ndcg@10"], report["test"]["all"]["recall@10"])